# r/soccer analytical pipeline on Google Colab (GPU)

**FIFA Communities chapter — `code/06-analysis-pipeline.py` run on a Colab GPU.** Orchestration only: every cell calls
the same command-line program that ran locally in the session-05 dry run; nothing analytical lives in this notebook.
Design: `docs/session-04_2026-10-22/01_analytical-pipeline-design.md` (§10 for Colab). Approvals, the local dry-run log
and the runbook: `docs/session-05_2026-10-23/`.

**Runtime:** *Runtime → Change runtime type → GPU*, A100 40 GB preferred (L4 acceptable; T4 works in float16, ≈ 3× slower),
high-RAM. In VS Code with the Colab extension: open this notebook, choose *New Colab Server → GPU*.

**Before the first run, once:** upload the folder `data/yearly datasets/` (17 CSV files, 2.4 GB) to your Google Drive as
`MyDrive/fifa-reddit-communities/data/yearly datasets/`. Results (`results/analysis/`, `data/analysis/`), the embedding
cache and the model weights (`.hf-cache`) are kept on Drive, so a disconnect costs nothing that is finished.

**Two passes.** *Pass 1 — smoke test on bin 1* (≈ 15–20 min of GPU time): every stage once, labelling only the 12 largest
topics, with a check cell after each stage that compares with the local dry run (92 topics, 35.9 % raw outliers, 6 categories).
*Pass 2 — the real run*: bin 1 in full, the gates, then bins 2–9. Order of operations (design §3): bins → embed + topics →
**label** → *co-author review* (`topic_overrides.csv`) → **taxonomy** → *review* (`category_overrides.csv`) → calibration of
the outlier threshold `T` → `reassign` → sentiment → collect → copy back → S6 locally in R.

## 0. Environment (run cells 0.1–0.4 every session; ≈ 3 min)

In [ ]:
# 0.1 Google Drive. This cell tests whether drive.mount works from this kernel (the VS Code Colab extension was not
#     documented on this point); the fallback downloads a shared Drive folder with gdown.
import os, subprocess, sys
DRIVE = '/content/drive/MyDrive/fifa-reddit-communities'    # adjust if you used another folder name
SHARED_DATA_URL = ''   # fallback only: a Drive *folder* link to 'data/yearly datasets' shared with the account
try:
    from google.colab import drive
    drive.mount('/content/drive')
    assert os.path.isdir(f'{DRIVE}/data/yearly datasets'), f'{DRIVE}/data/yearly datasets not found on Drive'
    n = len([f for f in os.listdir(f'{DRIVE}/data/yearly datasets') if f.endswith('.csv')])
    print('Drive mounted; yearly files on Drive:', n)
    assert n == 17, 'expected 17 yearly files'
except Exception as e:
    print('drive.mount failed or folder missing:', e)
    if not SHARED_DATA_URL:
        raise SystemExit('Set SHARED_DATA_URL to a shared Drive folder link and re-run this cell')
    os.makedirs(f'{DRIVE}/data', exist_ok=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
    subprocess.run(['gdown', '--folder', SHARED_DATA_URL, '-O', f'{DRIVE}/data/yearly datasets'], check=True)

In [ ]:
# 0.2 Code from GitHub (public repository), data copied from Drive to the local SSD (Drive I/O is slow for large CSVs),
#     results / caches / weights symlinked to Drive so they persist across sessions.
import os, subprocess, shutil
REPO = '/content/fifa-reddit-communities'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/alannemi/fifa-reddit-communities', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
os.chdir(REPO)
print('repository at', subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip())
os.makedirs('data', exist_ok=True)
if not os.path.isdir('data/yearly datasets'):
    shutil.copytree(f'{DRIVE}/data/yearly datasets', 'data/yearly datasets')
files = sorted(f for f in os.listdir('data/yearly datasets') if f.endswith('.csv'))
assert len(files) == 17, files
print('yearly files on local SSD:', len(files), '| total GB:', round(sum(os.path.getsize(f'data/yearly datasets/{f}') for f in files) / 1e9, 2))
for local, remote in [('results/analysis', f'{DRIVE}/results/analysis'), ('data/analysis', f'{DRIVE}/data/analysis'),
                      ('results/topic-models/embeddings', f'{DRIVE}/results/topic-models/embeddings'), ('.hf-cache', f'{DRIVE}/.hf-cache')]:
    os.makedirs(remote, exist_ok=True); os.makedirs(os.path.dirname(local) or '.', exist_ok=True)
    if os.path.islink(local):
        os.unlink(local)
    elif os.path.isdir(local):
        shutil.rmtree(local)
    os.symlink(remote, local)
os.environ['HF_HOME'] = f'{REPO}/.hf-cache'; os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('working in', os.getcwd(), '| results/analysis ->', os.path.realpath('results/analysis'))

In [ ]:
# 0.3 Environment: install ONLY the pinned packages on top of Colab's RAPIDS stack (review finding R8), then prove that
#     cuML, CUDA and the pins are intact. If `pip check` reports a broken requirement, fix that one package explicitly
#     (do not re-run a plain `pip install -r`).
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-r', 'code/requirements-colab.txt'], check=True)
chk = subprocess.run([sys.executable, '-m', 'pip', 'check'], capture_output=True, text=True)
print(chk.stdout.strip() or 'pip check: OK')
import torch, cuml, bertopic, umap, hdbscan, sentence_transformers, transformers, numpy, sklearn, pandas
assert torch.cuda.is_available(), 'no CUDA device — change the runtime type to GPU'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__, '| CUDA', torch.version.cuda, '| cuml', cuml.__version__)
print('bertopic', bertopic.__version__, '| umap', umap.__version__, '| hdbscan', hdbscan.__version__,
      '| sentence-transformers', sentence_transformers.__version__, '| transformers', transformers.__version__,
      '| numpy', numpy.__version__, '| sklearn', sklearn.__version__, '| pandas', pandas.__version__)
assert bertopic.__version__ == '0.17.4' and umap.__version__ == '0.5.12' and hdbscan.__version__ == '0.8.44', 'pins not honoured'
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.used', '--format=csv'])

In [ ]:
# 0.4 Helpers: run one stage (accel=True prefixes `python -m cuml.accel` for the GPU UMAP/HDBSCAN) and read a bin's config.json.
import subprocess, sys, shlex, time, json, os
import pandas as pd
def stage(args, accel=False):
    cmd = [sys.executable] + (['-m', 'cuml.accel'] if accel else []) + ['code/06-analysis-pipeline.py'] + shlex.split(args)
    print('>>>', ' '.join(cmd), flush=True); t0 = time.time()
    subprocess.run(cmd, check=True)
    print(f'<<< done in {(time.time() - t0) / 60:.1f} min', flush=True)
def cfg(b):
    return json.load(open(f'results/analysis/bin-{b}/config.json'))
def unit_checks():
    subprocess.run([sys.executable, 'code/tests/test_pipeline_units.py'], check=True)
unit_checks()          # 15 checks, ~1 min, no model needed — proves the code imports and behaves on this machine

## 1. S0 — period bins (every post; Σ must equal 1,302,021 and the nine counts the design table)

In [ ]:
stage('--stage bins')
s = pd.read_csv('data/analysis/period_bins_summary.csv')
print(s[['period_bin', 'period_type', 'n_posts', 'expected_n_design', 'min_cluster_size_start']].to_string(index=False))
assert (s.n_posts == s.expected_n_design).all() and s.n_posts.sum() == 1302021, 'bin sizes differ from the design table'

## 2. Pass 1 — smoke test on bin 1 (≈ 15–20 min of GPU time)
Every stage once on bin 1, with labelling limited to the 12 largest topics. Each check cell compares with the local dry run
of 2026-10-23 (MiniLM, CPU): 10,002 posts, 9,809 modelled, **92 topics at min_cluster_size 25, 35.9 % raw outliers**;
12 labelled topics, 1 needing review; 6 categories. With mpnet embeddings the numbers will differ somewhat — the check is
that the stages run on the GPU backend and the orders of magnitude match. Outputs go to the real `results/analysis/bin-1/`;
Pass 2 resumes from them (labelling continues from `labels.jsonl`; the taxonomy is re-run with `--force`).

In [ ]:
stage('--stage embed --bin 1')
stage('--stage topics --bin 1', accel=True)
c = cfg(1)['topics']
print(json.dumps({k: c[k] for k in ['n_documents', 'n_modelled', 'n_topics_raw', 'outlier_share_raw', 'elapsed_min']}, indent=1))
print('backend:', c['backend']); print('guard:', c['guard'])
assert c['backend']['backend'] == 'cuml', 'UMAP/HDBSCAN did not run on cuML — check that the command was prefixed with python -m cuml.accel'
assert c['guard']['converged'], 'topic-count guard did not reach 50–100 topics: see guard trials above (design §6.2, dry-run finding D1)'
print('calibration candidates (T, n reassigned):', [(r['threshold'], r['n_reassigned_at_T']) for r in c['calibration']])

In [ ]:
stage('--stage label --bin 1 --max-topics 12')
t = pd.read_csv('results/analysis/bin-1/topics_labelled.csv')
print(t[['topic_id', 'name', 'tone_category', 'round_iterations', 'agreement_mode', 'needs_review']].to_string(index=False))
l = cfg(1)['label']; print('llm:', l['llm_model'], l['llm_revision'], l['dtype'], 'batch', l['batch_size'], '| stats', l['llm_stats'], '| min', l['elapsed_min'])
assert len(t) == 12 and l['llm_stats']['truncated'] == 0

In [ ]:
stage('--stage taxonomy --bin 1')
c = pd.read_csv('results/analysis/bin-1/categories.csv')
print(c[['category_id', 'name', 'tone_category', 'topic_ids', 'n_topics', 'n_posts', 'agreement_mode', 'round_iterations', 'n_repaired']].to_string(index=False))
tj = json.load(open('results/analysis/bin-1/taxonomy.json'))
print('answers recovered by the JSON salvage (N11):', tj['n_answers_salvaged'], '| unplaced:', sum(v == 'unplaced' for v in tj['category_assignment'].values()))
assert len(c) >= 1, 'no category produced — read taxonomy.json'

In [ ]:
stage('--stage sentiment --years 2010')      # the 2010 file only, for the smoke test; Pass 2 scores every year
stage('--stage collect --allow-partial')
side = json.load(open('data/analysis/post_variables.json'))
print(json.dumps({k: side[k] for k in ['n_posts', 'bins_complete', 'complete', 'n_with_sentiment', 'n_with_topic', 'n_with_category', 'errors']}, indent=1))
assert side['n_posts'] == 1302021 and side['bins_complete'] == [1] and not side['errors']
print('Pass 1 complete — open results/analysis/bin-1/report.html (on Drive) and compare with the local dry run.')

## 3. Pass 2 — the real run
### 3.1 Bin 1 in full — gate 1 (design §12.1) and gate 4 (§12.4)
Resumes from Pass 1: the 80 remaining topics are labelled; then the taxonomy over all topics. Read `report.html` (2010 themes:
vuvuzelas, Jabulani, Lampard's goal, Suárez's handball, Spain's win, Paul the octopus) and the label summary
(`needs_review` ≤ 15 %, the `agreement_mode` and round distributions).

In [ ]:
stage('--stage label --bin 1')                 # resumable; re-run the cell after a disconnect
t = pd.read_csv('results/analysis/bin-1/topics_labelled.csv')
print(f'{len(t)} topics | needs_review {t.needs_review.sum()} ({t.needs_review.mean():.0%}) | modes {t.agreement_mode.value_counts().to_dict()} | rounds {t.round_iterations.value_counts().sort_index().to_dict()}')
stage('--stage taxonomy --bin 1 --force')     # all topics now
c = pd.read_csv('results/analysis/bin-1/categories.csv')
print(f'{len(c)} categories [{c.agreement_mode.iloc[0]}, {c.round_iterations.iloc[0]} round(s)], repaired topics {c.n_repaired.sum()}')
print(c[['category_id', 'name', 'n_topics', 'n_posts', 'share']].to_string(index=False))

### 3.2 Optional comparisons on bin 1 (design §12.4) — each into its own folder, so bin 1's real outputs stay clean

In [ ]:
RUN_COMPARISONS = False
if RUN_COMPARISONS:
    import shutil
    for name, extra in [('cmp-seeded', '--sampling seeded'), ('cmp-4b', '--llm Qwen/Qwen3-4B')]:
        root = f'/content/{name}'
        os.makedirs(f'{root}/results/analysis', exist_ok=True); os.makedirs(f'{root}/data/analysis', exist_ok=True)
        if not os.path.isdir(f'{root}/results/analysis/bin-1'):
            shutil.copytree('results/analysis/bin-1', f'{root}/results/analysis/bin-1', ignore=shutil.ignore_patterns('labels*', 'topics_labelled.csv', 'taxonomy.json', 'categories.csv', 'topic_categories.csv', '*_overrides.csv'))
            shutil.copy('data/analysis/period_bins.csv', f'{root}/data/analysis/')
        stage(f'--stage label --bin 1 {extra} --out-root {root}')
        t2 = pd.read_csv(f'{root}/results/analysis/bin-1/topics_labelled.csv')
        print(name, '| needs_review', t2.needs_review.mean().round(3), '| modes', t2.agreement_mode.value_counts().to_dict())

### 3.3 Large-bin gate (design §12.2): bin 4 (331k posts) or bin 6 (306k)
Raw outlier share ≤ 45 %, guard converged within four trials, minutes on the GPU. If the gate fails, revisit `HDBSCAN_MIN_SAMPLES`
and/or `TOPIC_COUNT_RANGE` (a decision for the design's §14) before any other bin.

In [ ]:
GATE_BIN = 4
stage(f'--stage embed --bin {GATE_BIN}')
stage(f'--stage topics --bin {GATE_BIN}', accel=True)
c = cfg(GATE_BIN)['topics']
print(json.dumps({k: c[k] for k in ['n_modelled', 'n_topics_raw', 'outlier_share_raw', 'elapsed_min']}, indent=1), '| guard', c['guard'], '| backend', c['backend']['backend'])
assert c['outlier_share_raw'] <= 0.45 and c['guard']['converged'], 'large-bin gate failed — see design §12.2'
stage(f'--stage label --bin {GATE_BIN}')
stage(f'--stage taxonomy --bin {GATE_BIN}')

### 3.4 Outlier threshold `T` (decision D3; gate §12.3)
`results/analysis/bin-N/outlier_calibration/` holds the quantile table and one 100-post judgement sample per candidate threshold
(`belongs` column for the co-author), for bin 1 and the large bin. Once `T` is chosen, record it in `config.py`
(`OUTLIER_THRESHOLD`) and in the design's decisions log, then apply it here (seconds per bin; never changes `topic_id_raw`,
keywords or labels — only sizes, prevalence and the report).

In [ ]:
T = None                        # e.g. 0.55 after the co-author check
if T is not None:
    for b in sorted({1, GATE_BIN}):
        stage(f'--stage reassign --bin {b} --outlier-threshold {T}')

### 3.5 Bins 2–9 (after the reviews of bin 1 and the gate bin): embed + topics → label → *review* → taxonomy → *review* → reassign

In [ ]:
BINS = [b for b in range(2, 10) if b != GATE_BIN]
for b in BINS:
    stage(f'--stage embed --bin {b}')
    stage(f'--stage topics --bin {b}', accel=True)
    c = cfg(b)['topics']; print(f"bin {b}: {c['n_topics_raw']} topics, raw outliers {c['outlier_share_raw']:.1%}, backend {c['backend']['backend']}, guard converged {c['guard']['converged']}")
for b in BINS:
    stage(f'--stage label --bin {b}')
# --- review pause: topic_overrides.csv per bin --- then:
# for b in BINS: stage(f'--stage taxonomy --bin {b}')
# --- review pause: category_overrides.csv per bin --- then reassign with the fixed T:
# for b in BINS: stage(f'--stage reassign --bin {b} --outlier-threshold {T}')

## 4. S4 sentiment (every year, CPU, ≈ 15 min) and S5 collect

In [ ]:
stage('--stage sentiment --force')
stage('--stage collect')          # without --allow-partial: must end with complete = true
side = json.load(open('data/analysis/post_variables.json'))
print(json.dumps({k: v for k, v in side.items() if k != 'versions'}, indent=1))
assert side['complete'], 'collection not complete — see bins_state above'

## 5. Copy back and finish locally
`results/analysis/` and `data/analysis/` are symlinked to Drive, so they are already there. Download them into the repository
(`results/analysis/` is tracked in git; `data/analysis/` goes to the shared data folder), then run stage S6 locally:
```bash
Rscript code/07-tag-corpus.R && Rscript code/08-validate-analysis.R
```
and finalise codebook v3.0 (`docs/session-05_2026-10-23/03_runbook-session-06.md`).

In [ ]:
# inventory of what is on Drive now
for root in ['results/analysis', 'data/analysis']:
    for dirpath, dirs, files in os.walk(root):
        if dirpath.count('/') <= 2:
            print(dirpath, len(files), 'files')